In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS movie_gold.results_movie(
    year_release_date INT,
    country_name STRING,
    company_name STRING,
    budget FLOAT,
    revenue FLOAT,
    movie_id INT,
    country_id INT,
    company_id INT,
    created_date DATE,
    updated_date DATE
    )
    USING DELTA
    """)


In [0]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW V_results_movie
    AS
    select m.year_release_date, c.country_name, pco.company_name, 
        m.budget, m.revenue, m.movie_id, c.country_id,pco.company_id
    from movie_silver.movies as m 
    inner join movie_silver.production_country as pc on m.movie_id = pc.movie_id
    inner join movie_silver.countries as c on pc.country_id = c.country_id
    inner join movie_silver.movie_company as mc on m.movie_id = mc.movie_id
    inner join movie_silver.production_company as pco on mc.company_id = pco.company_id
    where m.file_date ='{v_file_date}'
""")

In [0]:
spark.sql(f"""
            MERGE INTO movie_gold.results_movie tgt
            USING V_results_movie src
            ON (tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id )
            WHEN MATCHED THEN
                UPDATE SET 
                tgt.year_release_date = src.year_release_date,
                tgt.country_name = src.country_name,
                tgt.company_name = src.company_name,
                tgt.budget = src.budget,
                tgt.revenue = src.revenue,
                tgt.updated_date = current_timestamp
            WHEN NOT MATCHED THEN
                INSERT (year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id, created_date)
                VALUES (year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id,current_timestamp)
    """)

In [0]:
%sql
select count(1) from V_results_movie

In [0]:
%sql
select count(1) from movie_gold.results_movie